# Get the data

In [58]:
import pandas as pd

# Get the original dataset
df = pd.read_csv('../../../datasets/housing.csv')

# Get the dataset after EDA
df_prepared = df[['longitude', 'latitude', 'housing_median_age', 'total_bedrooms', 'median_income', 'median_house_value']]

In [59]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [60]:
df_prepared.head()

,longitude,latitude,housing_median_age,total_bedrooms,median_income,median_house_value
0,-122.23,37.88,41.0,129.0,8.3252,452600.0
1,-122.22,37.86,21.0,1106.0,8.3014,358500.0
2,-122.24,37.85,52.0,190.0,7.2574,352100.0
3,-122.25,37.85,52.0,235.0,5.6431,341300.0
4,-122.25,37.85,52.0,280.0,3.8462,342200.0


# Preprocessing and SPLIT

In [61]:
# ORIGINAL DATASET


from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split


# Preprocessing ORGINAL
encoder = OrdinalEncoder()
df['ocean_proximity'] = encoder.fit_transform(df[['ocean_proximity']])
df['longitude'] = df['longitude'].abs()
df.dropna(inplace = True)

# Split the data 
X = df.drop('median_house_value', axis = 1)
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

print(f"Train shapes: X:{X_train.shape}, y: {y_train.shape}")
print(f"Test shape: X: {X_test.shape}, {y_test.shape}")

Train shapes: X:(16346, 9), y: (16346,)
Test shape: X: (4087, 9), (4087,)


In [62]:
# Preprocessing PREPARED
df_prepared.dropna(inplace = True)


# PREPARED DATASET

X_prepared = df_prepared.drop('median_house_value', axis = 1)
y_prepared = df_prepared['median_house_value']

X_train_prepared, X_test_prepared, y_train_prepared, y_test_prepared = train_test_split(X_prepared, y_prepared, test_size= 0.2, random_state= 42)

print(f"Train shapes: X:{X_train_prepared.shape}, y: {y_train_prepared.shape}")
print(f"Test shape: X: {X_test_prepared.shape}, {y_test_prepared.shape}")

Train shapes: X:(16346, 5), y: (16346,)
Test shape: X: (4087, 5), (4087,)


C:\Users\tevos\AppData\Local\Temp\ipykernel_3096\1134795254.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prepared.dropna(inplace = True)


# Train model and see metrics

In [63]:
import warnings
warnings.simplefilter("ignore")

In [64]:
# LIBS

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, mean_squared_error

In [65]:
# ORIGINAL DATASET

models = [LinearRegression(), DecisionTreeRegressor(), RandomForestRegressor(n_estimators= 100, max_depth= 10), KNeighborsRegressor(n_neighbors= 5), XGBRegressor(), LGBMRegressor()]

for model in models:
    print('----------------------------------------')
    print(f"Model: {model}")

    pipe = Pipeline(
        steps=[
            ("Scaler", StandardScaler()),
            ("Model", model)
        ]
    )

    # TRAIN THE MODEL
    pipe.fit(X_train, y_train)

    # See if overfitting (predict the train pipe)
    y_preds_train = pipe.predict(X_train)
    mae = mean_absolute_error(y_train, y_preds_train)
    r2 = r2_score(y_train, y_preds_train)
    rmse = root_mean_squared_error(y_train, y_preds_train)
    mse = mean_squared_error(y_train, y_preds_train)
    print(f"Train Metrics: MAE:{mae} | R2: {r2*100:.2f}% | RMSE: {rmse} | MSE: {mse}")

    # See the metrics (test set)
    y_preds_test = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, y_preds_test)
    r2 = r2_score(y_test, y_preds_test)
    rmse = root_mean_squared_error(y_test, y_preds_test)
    mse = mean_squared_error(y_test, y_preds_test)
    print(f"Train Metrics: MAE:{mae} | R2: {r2*100:.2f}% | RMSE: {rmse} | MSE: {mse}")
    

----------------------------------------
Model: LinearRegression()
Train Metrics: MAE:50630.497436579884 | R2: 63.60% | RMSE: 69409.6962582108 | MSE: 4817705934.657082
Train Metrics: MAE:51388.70018950774 | R2: 63.99% | RMSE: 70171.99539639697 | MSE: 4924108937.911958
----------------------------------------
Model: DecisionTreeRegressor()
Train Metrics: MAE:0.0 | R2: 100.00% | RMSE: 0.0 | MSE: 0.0
Train Metrics: MAE:43468.01296794715 | R2: 65.93% | RMSE: 68257.00826462686 | MSE: 4659019177.237338
----------------------------------------
Model: RandomForestRegressor(max_depth=10)
Train Metrics: MAE:28952.684698901605 | R2: 86.84% | RMSE: 41737.72497134777 | MSE: 1742037685.7838671
Train Metrics: MAE:36196.356262150075 | R2: 78.67% | RMSE: 54014.0145877754 | MSE: 2917513771.8884134
----------------------------------------
Model: KNeighborsRegressor()
Train Metrics: MAE:33082.66922794568 | R2: 81.51% | RMSE: 49472.864607646974 | MSE: 2447564332.4865685
Train Metrics: MAE:41316.98894054319

In [66]:
# PREPARED DATASET

models = [LinearRegression(), DecisionTreeRegressor(), RandomForestRegressor(n_estimators= 150, max_depth= 10), KNeighborsRegressor(n_neighbors= 5), XGBRegressor(), LGBMRegressor()]

for model in models:
    print('----------------------------------------')
    print(f"Model: {model}")

    pipe = Pipeline(
        steps=[
            ("Scaler", StandardScaler()),
            ("Model", model)
        ]
    )

    # TRAIN THE MODEL
    pipe.fit(X_train_prepared, y_train_prepared)

    # See if overfitting (predict the train pipe)
    y_preds_train = pipe.predict(X_train_prepared)
    mae = mean_absolute_error(y_train_prepared, y_preds_train)
    r2 = r2_score(y_train_prepared, y_preds_train)
    rmse = root_mean_squared_error(y_train_prepared, y_preds_train)
    mse = mean_squared_error(y_train_prepared, y_preds_train)
    print(f"Train Metrics: MAE:{mae} | R2: {r2*100:.2f}% | RMSE: {rmse} | MSE: {mse}")

    # See the metrics (test set)
    y_preds_test = pipe.predict(X_test_prepared)
    mae = mean_absolute_error(y_test_prepared, y_preds_test)
    r2 = r2_score(y_test_prepared, y_preds_test)
    rmse = root_mean_squared_error(y_test_prepared, y_preds_test)
    mse = mean_squared_error(y_test_prepared, y_preds_test)
    print(f"Train Metrics: MAE:{mae} | R2: {r2*100:.2f}% | RMSE: {rmse} | MSE: {mse}")
    

----------------------------------------
Model: LinearRegression()


Train Metrics: MAE:53592.18528295427 | R2: 60.26% | RMSE: 72526.1050113491 | MSE: 5260035908.117236
Train Metrics: MAE:54392.80734844221 | R2: 60.29% | RMSE: 73691.06137945506 | MSE: 5430372527.230613
----------------------------------------
Model: DecisionTreeRegressor()
Train Metrics: MAE:0.0 | R2: 100.00% | RMSE: 0.0 | MSE: 0.0
Train Metrics: MAE:43475.96452165402 | R2: 65.06% | RMSE: 69119.6222110214 | MSE: 4777522174.594323
----------------------------------------
Model: RandomForestRegressor(max_depth=10, n_estimators=150)
Train Metrics: MAE:29733.394260177025 | R2: 86.10% | RMSE: 42893.209354216015 | MSE: 1839827408.7046041
Train Metrics: MAE:36332.50407280474 | R2: 78.53% | RMSE: 54181.16896929511 | MSE: 2935599070.8793077
----------------------------------------
Model: KNeighborsRegressor()
Train Metrics: MAE:37107.57258044781 | R2: 77.74% | RMSE: 54288.03416473552 | MSE: 2947190653.4714913
Train Metrics: MAE:46028.20063616345 | R2: 66.20% | RMSE: 67988.34969607205 | MSE: 4622